# Synthetic Data Generator

In [ ]:
%pip install -q \
    "transformers" \
    "gradio" \
    "jsonschema" \
    pandas

In [ ]:
# imports
import json
import math
import tempfile

from datetime import datetime, timezone, date
from pathlib import Path
from huggingface_hub import get_token

import pandas as pd
import torch
import gradio as gr
import gc

from jsonschema import Draft202012Validator, FormatChecker
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

In [ ]:
# GPU Check
if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime → Change runtime type → GPU."
    )

DEVICE = "cuda"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Gradio:", gr.__version__)

HF_TOKEN = get_token()

print(
    "Hugging Face authentication configured."
    if HF_TOKEN
    else "No token configured; using public model downloads."
)

In [ ]:
# Load Model

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" # "SmolLM2 1.7B": "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)

In [ ]:
# Define default dataset (Customer reviews)

DEFAULT_DESCRIPTION = """
Create fictional customer reviews of everyday consumer products.

Include varied products, writing styles, and positive, neutral,
and negative experiences. Reviews should sound natural.

Do not include customer names, email addresses, or other personal details.
"""

DEFAULT_RULES = """
- Ratings 1–2 should have negative sentiment.
- Rating 3 should have neutral sentiment.
- Ratings 4–5 should have positive sentiment.
- The review text should agree with the rating and sentiment.
- Avoid repeating the same review with minor wording changes.
"""

DEFAULT_SCHEMA = {
    "type": "object",

    "properties": {
        "product": {
            "type": "string",
            "minLength": 1,
            "maxLength": 60,
            "description": "A fictional or generic consumer product."
        },

        "category": {
            "type": "string",
            "enum": [
                "electronics",
                "home",
                "clothing",
                "sports"
            ]
        },

        "rating": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
        },

        "review": {
            "type": "string",
            "minLength": 15,
            "maxLength": 200,
            "description": "A short customer review."
        },

        "sentiment": {
            "type": "string",
            "enum": [
                "positive",
                "neutral",
                "negative"
            ]
        }
    },

    # Every record must contain all five columns.
    "required": [
        "product",
        "category",
        "rating",
        "review",
        "sentiment"
    ],

    # Reject unexpected columns.
    "additionalProperties": False
}

print(json.dumps(DEFAULT_SCHEMA, indent=2))

In [ ]:
# Support ticket and claims sample

def make_row_schema(properties):
    """Create a schema requiring all defined columns."""

    return {
        "type": "object",
        "properties": properties,
        "required": list(properties),
        "additionalProperties": False,
    }


TICKET_SCHEMA = make_row_schema({
    "ticket": {
        "type": "string",
        "minLength": 15,
        "maxLength": 200,
    },
    "category": {
        "type": "string",
        "enum": ["account", "billing", "technical"],
    },
    "priority": {
        "type": "string",
        "enum": ["low", "medium", "high"],
    },
})


CLAIMS_SCHEMA = make_row_schema({
    "member_id": {
        "type": "string",
        "enum": ["MEM001", "MEM002", "MEM003", "MEM004"],
    },
    "provider_id": {
        "type": "string",
        "enum": ["PROV001", "PROV002", "PROV003"],
    },
    "service_date": {
        "type": "string",
        "format": "date",
        "description": "Service date in YYYY-MM-DD format.",
    },
    "paid_date": {
        "type": "string",
        "format": "date",
        "description": "Payment date in YYYY-MM-DD format.",
    },
    "service_category": {
        "type": "string",
        "enum": ["assessment", "counseling", "group_session"],
    },
    "units": {
        "type": "integer",
        "minimum": 1,
        "maximum": 8,
    },
    "billed_cents": {
        "type": "integer",
        "minimum": 1000,
        "maximum": 100000,
    },
    "paid_cents": {
        "type": "integer",
        "minimum": 0,
        "maximum": 100000,
    },
})


PRESETS = {
    "Product reviews": {
        "description": DEFAULT_DESCRIPTION.strip(),
        "schema": DEFAULT_SCHEMA,
        "rules": DEFAULT_RULES.strip(),
    },

    "Support tickets": {
        "description": (
            "Create fictional customer support tickets for a software "
            "company. Include account, billing, and technical problems."
        ),
        "schema": TICKET_SCHEMA,
        "rules": (
            "The category should match the ticket text. "
            "Vary urgency and writing style. Do not include personal details."
        ),
    },

    "Simplified claims": {
        "description": (
            "Create fictional behavioral-health paid service records "
            "for software testing. These are simplified synthetic records, "
            "not actual claims."
        ),
        "schema": CLAIMS_SCHEMA,
        "rules": (
            "Use service dates during January–March 2025. "
            "Payment must occur on or after the service date. "
            "Paid cents must not exceed billed cents. "
            "Vary members, providers, services, units, and amounts."
        ),
    },
}


def load_preset(preset_name):
    """Populate the UI with properly serialized JSON."""

    preset = PRESETS[preset_name]

    return (
        preset["description"],
        json.dumps(preset["schema"], indent=2),
        preset["rules"],
    )


def claims_problem(record):
    """
    Return a validation problem, or None.

    These checks apply when the relevant claims columns are present.
    """

    if {"service_date", "paid_date"} <= record.keys():
        try:
            service_date = date.fromisoformat(record["service_date"])
            paid_date = date.fromisoformat(record["paid_date"])
        except (ValueError, TypeError):
            return "Service and payment dates must be valid ISO dates."

        if paid_date < service_date:
            return "Payment date cannot precede service date."

    if {"billed_cents", "paid_cents"} <= record.keys():
        if record["paid_cents"] > record["billed_cents"]:
            return "Paid cents cannot exceed billed cents."

    return None

In [ ]:
# Reuseable Function

SYSTEM_PROMPT = """
You generate fictional synthetic records for software testing and
machine-learning prototypes.

Follow the supplied schema and dataset requirements.

Return only a valid JSON array of objects.
Do not include Markdown, explanations, or text before or after the array.

Use JSON numbers for numeric fields and JSON booleans for boolean fields.
Do not add fields that are absent from the schema.
"""


def generate_text(messages, temperature=0.7, max_new_tokens=1500):
    """Generate a complete model response."""

    # Apply the conversation format expected by the model.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE)

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=float(temperature),
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    # A causal model returns input tokens followed by generated tokens.
    # Decode only the newly generated portion.
    generated_tokens = outputs[0, input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

In [ ]:
# Parse and validate the output

def prepare_schema(schema_text):
    """Parse the schema and report useful syntax errors."""

    try:
        schema = json.loads(schema_text)

    except json.JSONDecodeError as error:
        lines = schema_text.splitlines()

        offending_line = (
            lines[error.lineno - 1]
            if error.lineno <= len(lines)
            else ""
        )

        pointer = " " * (error.colno - 1) + "^"

        raise ValueError(
            f"JSON syntax error at line {error.lineno}, "
            f"column {error.colno}: {error.msg}\n"
            f"{offending_line}\n"
            f"{pointer}\n"
            "Use double quotes and JSON literals: false, true, null."
        ) from None

    Draft202012Validator.check_schema(schema)

    properties = schema.get("properties", {})

    if schema.get("type") != "object":
        raise ValueError("The schema must describe an object.")

    if not 1 <= len(properties) <= 8:
        raise ValueError("Use between 1 and 8 columns.")

    if set(schema.get("required", [])) != set(properties):
        raise ValueError("List every column under 'required'.")

    if schema.get("additionalProperties") is not False:
        raise ValueError("Set 'additionalProperties' to false.")

    allowed_types = {"string", "integer", "number", "boolean"}

    for name, definition in properties.items():
        if definition.get("type") not in allowed_types:
            raise ValueError(
                f"{name}: use string, integer, number, or boolean."
            )

    # FormatChecker enables validation of fields such as ISO dates.
    validator = Draft202012Validator(
        schema,
        format_checker=FormatChecker(),
    )

    return schema, validator


def parse_records(raw_text):
    """Parse a JSON array, tolerating a surrounding Markdown code fence."""

    text = raw_text.strip()

    # Some models might include a code fence despite the instructions.
    if text.startswith("```") and text.endswith("```"):
        text = "\n".join(text.splitlines()[1:-1]).strip()

    records = json.loads(text)

    if not isinstance(records, list):
        raise ValueError("Expected a JSON array of records.")

    return records


def record_key(record):
    """Create a stable representation for detecting exact duplicates."""

    return json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    )

In [ ]:
# Generation Pipeline

def generate_dataset(
    description,
    schema_text,
    rules,
    row_count,
    temperature,
    seed,
    progress=gr.Progress(),
):
    """Generate, validate, deduplicate, and export a synthetic dataset."""

    # Validate inputs before spending GPU time.
    try:
        schema, validator = prepare_schema(schema_text)
    except Exception as error:
        raise gr.Error(f"Invalid schema: {error}") from None

    if not description.strip():
        raise gr.Error("Describe the dataset you want to create.")

    if len(schema_text) > 6000:
        raise gr.Error("Please use a shorter schema.")

    row_count = int(row_count)
    seed = int(seed)
    temperature = float(temperature)

    if not 1 <= row_count <= 100:
        raise gr.Error("Request between 1 and 100 rows.")

    set_seed(seed)

    accepted = []
    seen = set()

    rejected = 0
    duplicates = 0
    parse_failures = 0
    attempts_used = 0
    business_rejected = 0

    last_raw_output = ""
    last_problem = ""

    batch_size = 3

    # Bound retries so generation cannot continue indefinitely.
    max_attempts = math.ceil(row_count / batch_size) * 3

    for attempt in range(max_attempts):
        if len(accepted) >= row_count:
            break

        attempts_used = attempt + 1
        remaining = row_count - len(accepted)
        requested_batch = min(batch_size, remaining)

        progress(
            len(accepted) / row_count,
            desc=f"Accepted {len(accepted)} of {row_count} rows",
        )

        # A few previous records provide a small novelty reminder.
        recent_records = accepted[-3:]

        user_prompt = f"""
Generate exactly {requested_batch} NEW fictional records.

DATASET DESCRIPTION:
{description}

SCHEMA FOR EACH RECORD:
{json.dumps(schema, ensure_ascii=False)}

ADDITIONAL REQUIREMENTS:
{rules}

Do not repeat these recent records:
{json.dumps(recent_records, ensure_ascii=False)}

Previous validation problem, if any:
{last_problem or "None"}

Return only a JSON array.
"""

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        try:
            last_raw_output = generate_text(
                messages,
                temperature=temperature,
            )
        except Exception as error:
            raise gr.Error(
                f"Model generation failed: {type(error).__name__}. "
                "Check the runtime; reduce output length if GPU memory is exhausted."
            ) from None

        # Malformed JSON gets another bounded attempt.
        try:
            candidates = parse_records(last_raw_output)
        except (ValueError, TypeError) as error:
            parse_failures += 1
            last_problem = str(error)[:250]
            continue

        if not candidates:
            last_problem = "The array was empty."
            continue

        for record in candidates:
            if len(accepted) >= row_count:
                break

            # Check required fields, types, ranges, and allowed values.
            errors = list(validator.iter_errors(record))

            if errors:
                rejected += 1
                last_problem = errors[0].message[:250]
                continue

            problem = claims_problem(record)

            if problem:
                business_rejected += 1
                last_problem = problem
                continue

            try:
                key = record_key(record)
            except (ValueError, TypeError):
                rejected += 1
                last_problem = "A record contained a non-JSON numeric value."
                continue

            if key in seen:
                duplicates += 1
                continue

            accepted.append(record)
            seen.add(key)

    columns = list(schema["properties"])
    dataframe = pd.DataFrame(accepted, columns=columns)

    complete = len(accepted) == row_count

    status = (
        f"{'Complete' if complete else 'Incomplete'}: "
        f"{len(accepted)}/{row_count} valid rows.\n"
        f"Attempts: {attempts_used}\n"
        f"Schema-rejected records: {rejected}\n"
        f"Exact duplicates removed: {duplicates}\n"
        f"Unparseable batches: {parse_failures}\n"
        f"Business-rule rejections: {business_rejected}"
    )

    if not complete:
        status += (
            "\nThe retry limit was reached. Try fewer columns, "
            "shorter text fields, or fewer requested rows."
        )

    # If nothing valid was produced, show diagnostics without empty downloads.
    if not accepted:
        return dataframe, status, [], last_raw_output

    # Each run gets its own folder so exports do not overwrite earlier runs.
    output_dir = Path(tempfile.mkdtemp(prefix="synthetic_data_"))

    csv_path = output_dir / "synthetic_data.csv"
    json_path = output_dir / "synthetic_data.json"
    metadata_path = output_dir / "generation_metadata.json"

    dataframe.to_csv(csv_path, index=False)

    json_path.write_text(
        json.dumps(accepted, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    metadata = {
        "synthetic": True,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model": MODEL_ID,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "description": description,
        "rules": rules,
        "schema": schema,
        "requested_rows": row_count,
        "generated_rows": len(accepted),
        "complete": complete,
        "temperature": temperature,
        "seed": seed,
        "attempts": attempts_used,
        "schema_rejections": rejected,
        "duplicates_removed": duplicates,
        "parse_failures": parse_failures,
        "business_rule_rejections": business_rejected,
    }

    metadata_path.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    return (
        dataframe,
        status,
        [str(csv_path), str(json_path), str(metadata_path)],
        last_raw_output,
    )

In [ ]:
# Check Validator

schema_text = json.dumps(DEFAULT_SCHEMA)

schema, validator = prepare_schema(schema_text)

valid_example = {
    "product": "Desk lamp",
    "category": "home",
    "rating": 5,
    "review": "Easy to assemble and bright enough for evening reading.",
    "sentiment": "positive",
}

invalid_example = {
    **valid_example,
    "rating": 8,
}

assert validator.is_valid(valid_example)
assert not validator.is_valid(invalid_example)

assert parse_records('[{"example": 1}]') == [{"example": 1}]

print("Basic parsing and validation checks passed.")

In [ ]:
# Using Gradio UI

# Close a previous UI if you rerun this cell.
if "demo" in globals():
    demo.close()


with gr.Blocks(
    title="Synthetic Data Generator",
    analytics_enabled=False,
    delete_cache=(3600, 3600),
) as demo:

    gr.Markdown(
        "# Synthetic Data Generator\n"
        "Create fictional datasets with an open model running in Colab."
    )

    with gr.Row():
        preset_choice = gr.Dropdown(
            choices=list(PRESETS),
            value="Product reviews",
            label="Dataset template",
        )

    description_box = gr.Textbox(
        value=DEFAULT_DESCRIPTION.strip(),
        label="Describe your dataset",
        lines=5,
    )

    with gr.Accordion("Column schema", open=True):
        schema_box = gr.Textbox(
            value=json.dumps(DEFAULT_SCHEMA, indent=2),
            label="JSON Schema for one record",
            lines=18,
        )

    rules_box = gr.Textbox(
        value=DEFAULT_RULES.strip(),
        label="Additional generation rules",
        lines=5,
    )

    with gr.Row():
        row_count = gr.Slider(
            minimum=1,
            maximum=100,
            value=12,
            step=1,
            label="Number of rows",
        )

        temperature = gr.Slider(
            minimum=0.1,
            maximum=1.0,
            value=0.7,
            step=0.1,
            label="Temperature",
        )

        seed = gr.Number(
            value=42,
            precision=0,
            label="Random seed",
        )

    generate_button = gr.Button(
        "Generate dataset",
        variant="primary",
    )

    status_box = gr.Textbox(
        label="Generation report",
        lines=6,
        interactive=False,
    )

    preview = gr.Dataframe(
        label="Generated records",
        interactive=False,
    )

    downloads = gr.File(
        label="Download CSV, JSON, and metadata",
        file_count="multiple",
    )

    with gr.Accordion("Inspect the latest raw model response", open=False):
        raw_output = gr.Textbox(
            label="Raw response",
            lines=12,
            interactive=False,
        )

    # Selecting a template fills the three configuration fields.
    preset_choice.change(
        fn=load_preset,
        inputs=preset_choice,
        outputs=[
            description_box,
            schema_box,
            rules_box,
        ],
    )

    generate_button.click(
        fn=generate_dataset,
        inputs=[
            description_box,
            schema_box,
            rules_box,
            row_count,
            temperature,
            seed,
        ],
        outputs=[
            preview,
            status_box,
            downloads,
            raw_output,
        ],
        concurrency_limit=1,
    )


demo.queue().launch(
    share=True,
    debug=True,
)